In [2]:
from google.cloud import storage
import requests
import os

def upload_pdfs_to_gcs(pdf_urls, bucket_name):
    """
    Download PDFs from URLs and upload them to Google Cloud Storage
    
    Args:
        pdf_urls (list): List of PDF URLs
        bucket_name (str): Name of the GCS bucket
    """
    # Initialize GCS client
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    for url in pdf_urls:
        try:
            # Download PDF
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            # Extract filename from URL
            filename = url.split('/')[-1]
            if not filename.endswith('.pdf'):
                filename += '.pdf'
            
            # Save PDF temporarily
            with open(filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            # Upload to GCS
            blob = bucket.blob(f'textbooks/{filename}')
            blob.upload_from_filename(filename)
            
            # Clean up local file
            os.remove(filename)
            
            print(f"Successfully uploaded {filename} to {bucket_name}")
            
        except Exception as e:
            print(f"Error processing {url}: {str(e)}")

# Example usage
pdf_urls = [
    'http://imlab.postech.ac.kr/dkim/class/csed514_2019s/DeepLearningBook.pdf',
    'https://d2l.ai/d2l-en.pdf',
    'https://arxiv.org/pdf/2301.04856',
    'http://www.incompleteideas.net/book/RLbook2020.pdf',
    'https://arxiv.org/pdf/2201.02135',
    'https://mml-book.github.io/book/mml-book.pdf',
    'https://alex.smola.org/drafts/thebook.pdf',
    'https://www.nrigroupindia.com/e-book/Introduction%20to%20Machine%20Learning%20with%20Python%20(%20PDFDrive.com%20)-min.pdf',
    'https://www.cmu.edu/intelligentbusiness/expertise/genai-principles.pdf',
    'https://ai.gov.ae/wp-content/uploads/2023/04/406.-Generative-AI-Guide_ver1-EN.pdf',
    'https://arxiv.org/pdf/2309.07930',
    'https://arxiv.org/pdf/2405.11029',
    
]
bucket_name = 'raw-files-mw'

upload_pdfs_to_gcs(pdf_urls, bucket_name)

Successfully uploaded DeepLearningBook.pdf to raw-files-mw
Successfully uploaded d2l-en.pdf to raw-files-mw
Successfully uploaded 2301.04856.pdf to raw-files-mw
Successfully uploaded RLbook2020.pdf to raw-files-mw
Successfully uploaded 2201.02135.pdf to raw-files-mw
Successfully uploaded mml-book.pdf to raw-files-mw
Successfully uploaded thebook.pdf to raw-files-mw
Successfully uploaded Introduction%20to%20Machine%20Learning%20with%20Python%20(%20PDFDrive.com%20)-min.pdf to raw-files-mw
Successfully uploaded genai-principles.pdf to raw-files-mw
Successfully uploaded 406.-Generative-AI-Guide_ver1-EN.pdf to raw-files-mw
Successfully uploaded 2309.07930.pdf to raw-files-mw
Successfully uploaded 2405.11029.pdf to raw-files-mw
